# 06. Feature importance and local explanations
Run notebook 03 first. Install the optional dependency with
`python -m pip install -e ".[dev,explain]"` in the notebook's environment.

SHAP explains the fitted Isolation Forest's **raw anomaly score**. Positive
contributions increase that score; they are neither fault probabilities nor causal
effects. They do not explain the statistical tier or incident hysteresis.
No importance cutoff, feature deletion, refitting or final-test access occurs.
Global importance uses a uniform sample of complete validation rows. High-score
examples are reported separately to avoid biasing global importance.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
from optical_anomaly.pipeline import prepare, develop, final_evaluation
from optical_anomaly.workflow import run_split

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
split = run_split(settings)
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [
    split.train_end,
    split.calibration_end,
    split.validation_end,
    split.test_end,
]
REPORT = RUN / "eda"
REPORT.mkdir(exist_ok=True)
print("Run:", RUN.resolve())

In [ ]:
from optical_anomaly.explanations import explain_run

FEATURE_SET = settings.get("model", {}).get("feature_set", "temperature")
# Other retained forests: rx_only, both_rx, tx_rx, fec, temperature.
EXPLANATIONS = explain_run(RUN, feature_set=FEATURE_SET)
metadata = json.loads((EXPLANATIONS / "metadata.json").read_text())
display(pd.Series(metadata))
importance = pd.read_csv(EXPLANATIONS / "importance.csv")
display(importance)  # Every feature, including zero importance.
importance.sort_values("mean_absolute_shap").plot.barh(
    x="feature", y="mean_absolute_shap", figsize=(10, 14), legend=False
)
plt.xlabel("Mean absolute SHAP contribution to raw anomaly score")
plt.tight_layout()
plt.show()

Correlated features can share attribution. Independent SHAP masking can also
create feature combinations absent from real telemetry. Inspect correlations and
local contributions together; low importance does not establish redundancy.
The default 64 validation examples and 16 healthy background rows are a small,
reproducible exploratory sample. Repeat with larger samples and different seeds
before interpreting the ranking as stable. There is no statistical significance
claim or automatic selection based on this ranking.

In [ ]:
correlations = pd.read_csv(EXPLANATIONS / "correlations.csv", index_col=0)
fig, ax = plt.subplots(figsize=(12, 10))
image = ax.imshow(correlations, vmin=-1, vmax=1, cmap="coolwarm")
fig.colorbar(image, ax=ax, label="Validation Spearman correlation")
ax.set_title("Feature dependence; all complete validation rows")
plt.show()

local = pd.read_csv(EXPLANATIONS / "high_score_shap.csv")
scores = pd.read_csv(EXPLANATIONS / "high_score_scores.csv")
display(scores)
row = local.iloc[0]
row.drop(["entity_id", "timestamp"]).astype(float).sort_values().plot.barh(
    figsize=(10, 14), title=f"Local attribution: {row.entity_id}, {row.timestamp}"
)
plt.xlabel("Signed SHAP contribution")
plt.tight_layout()
plt.show()
print("Reports:", EXPLANATIONS.resolve())

The saved score table verifies background score + feature contributions = actual
raw anomaly score. Report complete-score coverage alongside importance: unscored
rows are absent from these explanations. Review operational metrics in notebook 04;
SHAP does not measure early-detection utility or replace event-level evaluation.